In [4]:

import heapq

# Goal state
GOAL = (1, 2, 3,
        8, 0, 4,
        7, 6, 5)

# Initial state: one move away from the goal
START = (1, 2, 3,
         8, 4, 5,
         7, 0, 6)


# Generate neighboring states
def get_neighbors(state):
    neighbors = []
    blank = state.index(0)
    row, col = divmod(blank, 3)

    moves = [(-1, 0), (1, 0), (0, -1), (0, 1)]

    for dr, dc in moves:
        nr, nc = row + dr, col + dc

        if 0 <= nr < 3 and 0 <= nc < 3:
            new_pos = nr * 3 + nc
            new_state = list(state)

            new_state[blank], new_state[new_pos] = (
                new_state[new_pos], new_state[blank]
            )

            neighbors.append(tuple(new_state))

    return neighbors


# Heuristic 1: Misplaced Tiles
def misplaced_tiles(state):
    return sum(
        1 for i in range(9)
        if state[i] != 0 and state[i] != GOAL[i]
    )


# Heuristic 2: Manhattan Distance
def manhattan_distance(state):
    distance = 0

    for i, tile in enumerate(state):
        if tile == 0:
            continue

        current_row, current_col = divmod(i, 3)
        goal_pos = GOAL.index(tile)
        goal_row, goal_col = divmod(goal_pos, 3)

        distance += abs(current_row - goal_row)
        distance += abs(current_col - goal_col)

    return distance


# A* search
def astar(start, heuristic):
    priority_queue = [(heuristic(start), 0, start)]
    g_cost = {start: 0}
    parent = {start: None}

    while priority_queue:
        f, g, state = heapq.heappop(priority_queue)

        if g != g_cost[state]:
            continue

        if state == GOAL:
            path = []
            current = state

            while current is not None:
                path.append(current)
                current = parent[current]

            return path[::-1]

        for neighbor in get_neighbors(state):
            new_g = g + 1

            if new_g < g_cost.get(neighbor, float("inf")):
                g_cost[neighbor] = new_g
                parent[neighbor] = state

                new_f = new_g + heuristic(neighbor)
                heapq.heappush(
                    priority_queue,
                    (new_f, new_g, neighbor)
                )

    return None


# Display solution
def print_solution(path):
    if path is None:
        print("No solution found.")
        return

    print("Number of moves:", len(path) - 1)

    for step, state in enumerate(path):
        print(f"\nStep {step}:")
        for i in range(0, 9, 3):
            print(*state[i:i + 3])


# Run both heuristics
print("A* with Misplaced Tiles")
print_solution(astar(START, misplaced_tiles))

print("\nA* with Manhattan Distance")
print_solution(astar(START, manhattan_distance))


A* with Misplaced Tiles
Number of moves: 3

Step 0:
1 2 3
8 4 5
7 0 6

Step 1:
1 2 3
8 4 5
7 6 0

Step 2:
1 2 3
8 4 0
7 6 5

Step 3:
1 2 3
8 0 4
7 6 5

A* with Manhattan Distance
Number of moves: 3

Step 0:
1 2 3
8 4 5
7 0 6

Step 1:
1 2 3
8 4 5
7 6 0

Step 2:
1 2 3
8 4 0
7 6 5

Step 3:
1 2 3
8 0 4
7 6 5
